In [1]:
try:
    import optuna

except ModuleNotFoundError:
    import sys
    !{sys.executable} -m pip install optuna

In [11]:
import numpy as np
import pickle
import seaborn as sns
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor

from optuna import create_study, Trial

In [3]:
df = pd.read_csv("GFdata_binary.csv", sep = "\t")
display(df)

,ZnO,Sb2O3,Na2O,MoO3,Bi2O3,TeO2,V2O5,P2O5,B2O3,CaO,...,P2Se5,BiNbO4,Se4I,GeTe4.3,GaTe3,ZnI2,Na2Se,MgS,AsTe3,Classificacao
0,0.0,0.0,0.00,0.0,0.00,0.00,76.84,4.6,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
1,0.0,0.0,0.00,0.0,0.00,0.00,0.00,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
2,0.0,0.0,0.00,0.0,0.00,0.00,0.00,0.0,34.650000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
3,0.0,0.0,16.16,0.0,0.00,83.76,0.00,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
4,0.0,0.0,0.00,0.0,0.00,0.00,0.00,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50455,0.0,0.0,0.00,0.0,0.00,0.00,0.00,0.0,57.148686,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
50456,0.0,0.0,0.00,0.0,0.00,0.00,0.00,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
50457,0.0,0.0,0.00,0.0,0.00,0.00,0.00,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
50458,0.0,0.0,0.00,0.0,0.00,89.89,0.00,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1


In [4]:
TAMANHO_TESTE = 0.2
SEMENTE_ALEATORIA = 305
NUM_FOLDS = 10

DATASET_NAME = df
FEATURES = []
for column in df.columns:
    FEATURES.append(column)
FEATURES.remove("Classificacao")
print(FEATURES)

TARGET = ["Classificacao"]

NUM_TENTATIVAS = 50

['ZnO', 'Sb2O3', 'Na2O', 'MoO3', 'Bi2O3', 'TeO2', 'V2O5', 'P2O5', 'B2O3', 'CaO', 'Nb2O5', 'Ge', 'Se', 'Ag', 'BaO', 'K2O', 'SrO', 'Li2O', 'La2O3', 'MgO', 'ThO2', 'Al2O3', 'BeO', 'ZrO2', 'TiO2', 'Ta2O5', 'WO3', 'Tl2O', 'CdO', 'PbO', 'In2O3', 'SnO2', 'Ga2O3', 'Cs2O', 'Rb2O', 'Y2O3', 'Sc2O3', 'SiO2', 'GeO2', 'UO2', 'LaF3', 'ZrF4', 'PbF2', 'Ag2O', 'MnO', 'Cu2O', 'HfO2', 'AlN', 'CrO3', 'Nd2O3', 'AlF3', 'Fe2O3', 'Cr2O3', 'BaF2', 'InF3', 'MgF2', 'PbCl2', 'ZnCl2', 'PbI2', 'Fe3O4', 'CeO2', 'CuO', 'ThF4', 'ZnF2', 'YbF3', 'YF3', 'As2O3', 'Dy2O3', 'NiO', 'CoO', 'LiCl', 'SrF2', 'CaF2', 'Sm2O3', 'BeF2', 'KF', 'VO2', 'NaF', 'HgO', 'MnO2', 'As2O5', 'SO3', 'Na2S', 'MnF2', 'KCl', 'ThCl4', 'NaCl', 'BiCl3', 'CdCl2', 'PbBr2', 'CuCl', 'BaCl2', 'CsI', 'BaI2', 'CuI', 'ZnBr2', 'KBr', 'CdBr2', 'BaBr2', 'MgBr2', 'CaBr2', 'SrBr2', 'MnBr2', 'CoBr2', 'FeBr2', 'BiF3', 'CsF', 'AgF', 'AgI', 'FeO', 'TmF3', 'NdF3', 'TlBr', 'TlI', 'GdF3', 'SnO', 'SnCl2', 'Sb', 'Mn2O3', 'DyF3', 'HoF3', 'ErF3', 'Sb2O5', 'Gd2O3', 'SnF2', 'Er

In [5]:
df = df.reindex(FEATURES + TARGET, axis=1)
df = df.dropna()

indices = df.index
indices_treino, indices_teste = train_test_split(
    indices, test_size=TAMANHO_TESTE, random_state=SEMENTE_ALEATORIA
)

df_treino = df.loc[indices_treino]
df_teste = df.loc[indices_teste]

X_treino = df_treino.reindex(FEATURES, axis=1).values
y_treino = df_treino.reindex(TARGET, axis=1).values.ravel()

X_teste = df_teste.reindex(FEATURES, axis=1).values
y_teste = df_teste.reindex(TARGET, axis=1).values.ravel()

X = df.reindex(FEATURES, axis=1).values
y = df.reindex(TARGET, axis=1).values.ravel()

In [6]:
def cria_instancia_modelo(trial):
    """Cria uma instância do modelo.

    Args:
      trial: objeto tipo Trial do optuna.

    Returns:
      Uma instância do modelo desejado.
    """

    parametros = {
        "n_estimators": trial.suggest_int("n_estimators", 10, 100),

        "criterion": trial.suggest_categorical("criterion", ["squared_error", "friedman_mse", "poisson"]),

        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20, log=True),

        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20, log=True),

        "max_features": trial.suggest_float("max_features_internal", 0, 1),

        "n_jobs": -1,

        "bootstrap": True,

        "random_state": SEMENTE_ALEATORIA,
    }

    model = RandomForestRegressor(**parametros)

    return model

In [14]:
def funcao_objetivo(
    trial,
    X,
    y,
    num_folds=NUM_FOLDS,
):
    """Função objetivo do optuna

    Referencia:
      https://medium.com/@walter_sperat/ using-optuna-with-sklearn-the-right-way-part-1-6b4ad0ab2451
    """

    modelo = cria_instancia_modelo(trial)

    metricas = cross_val_score(
        modelo,
        X,
        y,
        scoring="accuracy",
        cv=NUM_FOLDS,
    )

    # releia sobre scores no sklearn para relembrar porque tem um negativo abaixo
    return metricas.mean()

In [15]:
objeto_de_estudo = create_study(direction="minimize")

[I 2024-04-02 13:46:36,151] A new study created in memory with name: no-name-15a90e4b-e23c-47c3-bac5-3b20c11b1e92


In [16]:
def funcao_objetivo_parcial(trial):
    return funcao_objetivo(trial, X_treino, y_treino)

In [ ]:
objeto_de_estudo.optimize(funcao_objetivo_parcial, n_trials=NUM_TENTATIVAS)

c:\venv\ilumpy\lib\site-packages\sklearn\model_selection\_validation.py:696: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "c:\venv\ilumpy\lib\site-packages\sklearn\model_selection\_validation.py", line 687, in _score
    scores = scorer(estimator, X_test, y_test)
  File "c:\venv\ilumpy\lib\site-packages\sklearn\metrics\_scorer.py", line 87, in __call__
    score = scorer._score(cached_call, estimator,
  File "c:\venv\ilumpy\lib\site-packages\sklearn\metrics\_scorer.py", line 242, in _score
    return self._sign * self._score_func(y_true, y_pred,
  File "c:\venv\ilumpy\lib\site-packages\sklearn\utils\validation.py", line 63, in inner_f
    return f(*args, **kwargs)
  File "c:\venv\ilumpy\lib\site-packages\sklearn\metrics\_classification.py", line 202, in accuracy_score
    y_type, y_true, y_pred = _check_targets(y_true, y_pred)
  File "c:\venv\ilumpy\lib\site-packages\skle